In [11]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import json
import os

# 1. Parameters

In [12]:
REGION = "wroclaw_small"
DATA_PATH = f"../data/{REGION}/clean.csv"
MODEL_PATH = f"../data/{REGION}/models/"
os.makedirs(MODEL_PATH, exist_ok=True)

targets = ['NDVI', 'NDWI', 'NDMI']
look_back = 5 # 5 how many days look back
forecast_steps = 6 * 3 # 6 * 5 = 30 so 1 month :3

# 2. Feature ingeninring

In [13]:
df = pd.read_csv(DATA_PATH)
df['Data'] = pd.to_datetime(df['Data'])
df = df.sort_values(['Sektor_ID', 'Data'])

for t in targets:
    df[f'target_{t}'] = df.groupby('Sektor_ID')[t].shift(-1)

df['day_of_year'] = df['Data'].dt.dayofyear
df['day_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
df['month'] = df['Data'].dt.month

lag_cols = []
for col in targets:
    for i in range(1, look_back + 1):
        col_name = f'{col}_lag_{i}'
        df[col_name] = df.groupby('Sektor_ID')[col].shift(i)
        lag_cols.append(col_name)

time_features = ['day_sin', 'day_cos', 'month']
final_features = ['NDVI', 'NDWI', 'NDMI', 'Lat', 'Lon'] + lag_cols + time_features
all_target_cols = [f'target_{t}' for t in targets]

df_ml = df.dropna(subset=all_target_cols + lag_cols).copy()

X = df_ml[final_features]
y = df_ml[all_target_cols]

X_train, X_test, y_train_all, y_test_all = train_test_split(X, y, test_size=0.2, random_state=42)


# 3. Model training

In [14]:

models = {}
for t in targets:
    print(f"📡 Training model for: {t}")
    y_train = y_train_all[f'target_{t}']
    y_test = y_test_all[f'target_{t}']
    
    m = xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        objective='reg:squarederror',
        random_state=42
    )
    
    m.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    # Eval
    y_pred = m.predict(X_test)
    print(f"   ✅ R2 Score: {r2_score(y_test, y_pred):.4f} | MAE: {mean_absolute_error(y_test, y_pred):.4f}")
    
    # Zapis
    m.save_model(f"{MODEL_PATH}model_{t.lower()}.json")
    models[t] = m

# meow ;3
#with open(f"../data/{REGION}/features_list.json", 'w') as f:
#    json.dump(final_features, f)


📡 Training model for: NDVI
   ✅ R2 Score: 0.9535 | MAE: 0.0206
📡 Training model for: NDWI
   ✅ R2 Score: 0.9349 | MAE: 0.0201
📡 Training model for: NDMI
   ✅ R2 Score: 0.9639 | MAE: 0.0153


In [15]:
current_batch = df.groupby('Sektor_ID').tail(1).copy()
recursive_results = []

for step in range(1, forecast_steps + 1):

    X_curr = current_batch[final_features]
    preds = {t: models[t].predict(X_curr) for t in targets}
    
    temp_res = current_batch[['Sektor_ID', 'Lat', 'Lon']].copy()
    for t in targets:
        temp_res[f'Pred_{t}'] = preds[t]
    
    forecast_date = current_batch['Data'].max() + pd.Timedelta(days=5 * step)
    temp_res['Forecast_Date'] = forecast_date
    temp_res['Step'] = step
    recursive_results.append(temp_res)

    for i in range(look_back, 1, -1):
        for t in targets:
            current_batch[f'{t}_lag_{i}'] = current_batch[f'{t}_lag_{i-1}']
    
    for t in targets:
        current_batch[f'{t}_lag_1'] = current_batch[t]
        
    for t in targets:
        current_batch[t] = preds[t]
        
    current_batch['day_of_year'] = forecast_date.dayofyear
    current_batch['day_sin'] = np.sin(2 * np.pi * current_batch['day_of_year'] / 365.25)
    current_batch['day_cos'] = np.cos(2 * np.pi * current_batch['day_of_year'] / 365.25)
    current_batch['month'] = forecast_date.month

forecast_df = pd.concat(recursive_results)
forecast_df.to_csv(f"../data/{REGION}/forecast_30d_recursive.csv", index=False)
print(f"✨ Success! Forecast saved to ../data/{REGION}/forecast_30d_recursive.csv")

✨ Success! Forecast saved to ../data/wroclaw_small/forecast_30d_recursive.csv
